# P122 — Síntesis de voz natural condicionando WaveNet con espectrogramas mel predichos

## 1. Título y paper

**Paper:** *Natural TTS Synthesis by Conditioning WaveNet on Mel Spectrogram Predictions*  
**Autoría:** Jonathan Shen, Ruoming Pang, Ron J. Weiss, Mike Schuster, Navdeep Jaitly, Zongheng Yang, Zhifeng Chen, Yu Zhang, Yuxuan Wang, RJ Skerry-Ryan, Rif A. Saurous, Yannis Agiomyrgiannakis, Yonghui Wu  
**Año y venue:** 2018 · ICASSP 2018, 4779–4783  
**Nivel:** L2 · **Motor:** `tacotron`  
**Ficha completa:** [`P122_tacotron`](../../papers/foundational/P122_tacotron/README.md)

**Hito:** Parte la síntesis en dos etapas con el espectrograma mel como interfaz, y alcanza naturalidad indistinguible de una grabación en la escala de opinión media.

- [doi:10.1109/ICASSP.2018.8461368](https://doi.org/10.1109/ICASSP.2018.8461368)

> Este notebook implementa una **miniatura** del mecanismo. No reproduce el experimento original ni sus métricas: reproduce la idea para que se pueda inspeccionar y discutir.


## 2. Objetivos

1. Explicar qué problema resolvió el paper: Predecir la forma de onda directamente desde el texto es intratable: tres segundos de audio son decenas de miles de pasos autorregresivos, y ningún modelo con atención puede alinear texto contra una secuencia de esa longitud.
2. Ejecutar una implementación mínima de la propuesta: Dos modelos con una interfaz explícita: uno predice el espectrograma mel desde el texto con atención, y un vocoder neuronal convierte ese espectrograma en forma de onda. Cada etapa se entrena y se sustituye por separado.
3. Predecir el resultado antes de ejecutar, y contrastar la predicción con la salida.
4. Identificar al menos una limitación de la miniatura y una del paper original.
5. Conectar el hito con el siguiente eslabón de la ruta.


## 3. Prerrequisitos

- Python 3.11+ y el paquete del programa instalado (`pip install -e .`).
- Haber leído la guía [método de lectura en 5 pasadas](../../papers/guides/METODO_DE_LECTURA_EN_5_PASADAS.md).
- Hitos previos:
- P07
- P119


## 4. Intuición

Nadie predice 66 150 muestras de audio una a una desde el texto. Pero 258 marcos de espectrograma sí — y de ahí a la forma de onda ya hay un modelo que sabe hacerlo.


## 5. Concepto mínimo

```text
texto ──[atención]──▶ espectrograma mel ──[vocoder]──▶ forma de onda
         etapa 1                  interfaz              etapa 2

3 s a 22 050 Hz = 66 150 muestras  →  258 marcos     (256× menos PASOS)
```


## 6. Código explicado

El motor aísla el mecanismo del paper con datos de juguete y salida inspeccionable.


In [ ]:
import json
import pathlib
import sys

ROOT = pathlib.Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from ai_evolution.papers_lab import run_paper_lab


def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


In [ ]:
r = run_paper_lab('tacotron', seed=7)['result']
show(r)

## 7. Predicción antes de ejecutar

1. ¿Cuánto comprime el espectrograma en número de valores?
2. ¿Y en número de pasos?
3. ¿Qué pasa si la atención se atasca?

> Escribe tu respuesta aquí antes de continuar.


## 8. Experimento controlado

Se varía una sola cosa y se observa el efecto.


In [ ]:
for semilla in (1, 7, 42):
    r = run_paper_lab('tacotron', seed=semilla)
    print(f'semilla {semilla:>2} · evidencia principal:')
    for e in r['evidence']:
        print('   +', e)
    break  # determinista: basta una para ver la estructura
for semilla in (1, 7, 42):
    r = run_paper_lab('tacotron', seed=semilla)['result']
    print(f'semilla {semilla:>2} → claves: {list(r)[:4]}')

## 9. Salida interpretable

En valores, solo **3,2×**. En pasos, **256×**: de 66 150 a **258**. Esa es la compresión que importa, porque el modelo es autorregresivo. Y si la atención se atasca en un carácter, la salida dice «hola que » en vez de «hola que tal»: se pierden **3 caracteres** del final.


## 10. Comentario pedagógico

El mel no ahorra memoria — ahorra **longitud de secuencia**, que es lo que hace tratable la atención. Y el modo de fallo que ves es el característico de la síntesis con atención: repetir o saltarse trozos. Por eso los sistemas posteriores fuerzan la monotonía en la arquitectura en vez de esperar que se aprenda.


## 11. Error o anti-patrón deliberado

Anti-patrón: evaluar un sintetizador por la pérdida de entrenamiento.


In [ ]:
print('La perdida sobre el espectrograma no correlaciona bien con lo que se oye.')
print('Un modelo con perdida baja puede saltarse una palabra entera.')
print('La metrica del articulo es la opinion media de oyentes humanos.')

## 12. Corrección

La aritmética de las dos etapas y el fallo de alineación:


In [ ]:
r = run_paper_lab('tacotron', seed=3)['result']
print('audio:', r['audio'])
print('espectrograma:', r['espectrograma'])
print('compresion en pasos:', r['compresion_en_pasos_de_tiempo'])
print('alineacion correcta:', r['alineacion_monotona'])
print('alineacion atascada:', r['alineacion_atascada'])

## 13. Desafío guiado

Explica por qué una interfaz explícita entre etapas permite sustituir el vocoder sin reentrenar la primera etapa, y qué se pierde a cambio frente a un modelo de extremo a extremo.


In [ ]:
r = run_paper_lab('tacotron', seed=3)['result']
show(r)

## 14. Desafío autónomo

Escucha dos sintetizadores con el mismo texto largo y busca repeticiones u omisiones. Anota en qué construcciones aparecen.


## 15. Evidencia de aprendizaje

Guarda los casos que encontraste y tu hipótesis sobre por qué fallan ahí.

Autoevaluación y respuestas esperadas: [ficha del paper](../../papers/foundational/P122_tacotron/README.md) · evaluación formal: [`assessments/papers/P122_tacotron.md`](../../assessments/papers/P122_tacotron.md)


## 16. Cierre

La voz sintética ya es indistinguible de una grabación. Eso abre el problema de los derechos sobre una identidad vocal, que es la ruta de medios.


## 17. Conexión con el siguiente hito

- P130

Ruta completa: [`papers/ROADMAP.md`](../../papers/ROADMAP.md)
